[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-08c-attention-heads.ipynb)

# Lecture 8c: what does each attention head do?

A Transformer layer does not run attention once; it runs it several times side by side, in
**heads**, and then joins the results. Here you write the step that makes heads out of one
layer's queries, keys and values, check it against a trained Transformer, and then read the
model's heads one at a time, on the English sentence it is reading.

By the end you will be able to:

1. split a layer's queries, keys and values into heads, and say why each head can attend
   somewhere different;
2. read self-attention head by head, and say what averaging the heads hides;
3. measure how sharp a layer's attention is, and say how it changes with depth.

These serve Lecture 8c's multi-head attention.

You write two short functions, marked TODO, and checks tell you when each is right; you write a
prediction, which stops the notebook until you write it, before the maps and the measurement
that test it. Everything runs on a CPU in under a minute. Lecture 8b's notebook used the same
model and sentence for cross-attention; this one stays inside the encoder.

**Running in Google Colab?** Run the cells in order. The second downloads the trained model,
about 11 MB. There is nothing to uncomment, and no GPU is needed.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — the in-class activity for Lecture 8c (*Finishing the Transformer, the A8 Debrief, and the Subword Activity*).

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo

In [ ]:
from torchlingo.colab import setup

# The trained model and its subword vocabulary. A pip install does not ship them, so setup
# downloads any that are missing.
env = setup(data=["data/pretrained/model.pt", "data/pretrained/spm.model"])

import math

import torch

from torchlingo.checkpoint import load_model
from torchlingo.data_processing.vocab import SentencePieceVocab
from torchlingo.visualization import format_attention

SENTENCE = "This report is our main public report."

# The model file holds the weights and the settings the model was built with; its vocabulary is
# the separate spm.model, one for both languages. load_model returns the model in inference
# mode: no dropout, so attention rows sum to 1.
sp_vocab = SentencePieceVocab("data/pretrained/spm.model")
transformer, _, _ = load_model("data/pretrained/model.pt", "cpu", src_vocab=sp_vocab)

source = torch.tensor([sp_vocab.encode(SENTENCE, add_special_tokens=True)])
source_pieces = sp_vocab.indices_to_tokens(source[0].tolist())
print("Pieces:", source_pieces)
print("Encoder layers:", len(transformer.transformer.encoder.layers),
      "| heads per layer:", transformer.transformer.encoder.layers[0].self_attn.num_heads,
      "| d_model:", transformer.transformer.encoder.layers[0].self_attn.embed_dim)

## Step 1: one projection, four heads

The model is a small Transformer of the kind you train for A8, trained on English–Spanish text.
It reads **subword pieces**: `▁` marks a piece that starts a word, so "report" is `▁re` +
`port`. In the encoder, attention is **self-attention**: the queries, keys and values all come
from the same sentence, so each piece looks at the other pieces of its own sentence.

Each layer makes a query, a key and a value for every piece by multiplying its input by three
learned matrices and adding a learned bias, Q = X W_Q + b_Q, K = X W_K + b_K, V = X W_V + b_V:
128 numbers per piece in this model. **Multi-head attention** then cuts each 128 into four
slices of 32, d_k = 128 / 4, and runs attention on each slice separately. Each head compares
pieces on its own 32 numbers, so each can look somewhere different; the layer joins the four
outputs back into 128 afterwards.

**Yours to write: `split_heads`.** It takes a (pieces, d_model) matrix and returns
(heads, pieces, d_k), where head h holds columns h·d_k up to (h+1)·d_k of every piece. Tensors
have `.reshape` and `.transpose`, and a slice such as `x[:, 2:4]` takes columns 2 and 3; any of
those will do. The check uses numbers you can follow by eye: work out on paper what head 1 should
be before you run it.

In [ ]:
def split_heads(x, n_heads):
    """Return x, shaped (pieces, d_model), as (n_heads, pieces, d_k), with d_k = d_model // n_heads.

    TODO: write it. Head h is columns h * d_k up to (h + 1) * d_k of every piece.
    """
    raise NotImplementedError("write split_heads, then run this cell again")


# Two pieces, d_model 6, three heads of d_k 2. Each number is its own position, so a misplaced
# one shows where it came from.
x = torch.arange(12.0).reshape(2, 6)
print(x)
heads = split_heads(x, 3)

assert tuple(heads.shape) == (3, 2, 2), (
    f"three heads, two pieces, d_k 2: expected shape (3, 2, 2), got {tuple(heads.shape)}"
)
assert torch.equal(heads[0], torch.tensor([[0.0, 1.0], [6.0, 7.0]])), (
    f"head 1 should be columns 0 and 1 of both pieces; got {heads[0].tolist()}"
)
assert torch.equal(heads[2], torch.tensor([[4.0, 5.0], [10.0, 11.0]])), (
    f"head 3 should be columns 4 and 5 of both pieces; got {heads[2].tolist()}"
)
print("split_heads passes its checks.")

## Step 2: your heads against the model's own

Each head then takes one ordinary attention step on its slice: softmax(QKᵀ / √d_k) V, the step
Lecture 8b computed by hand. You wrote it as `attention` in Lecture 8b's notebook; here it is
given. Because `@` works on the last two dimensions of a tensor, the same three lines run all
four heads at once.

The first cell records each encoder layer's input X, and the weights the model itself computed
for each head, while the model reads the sentence. The second makes Q, K and V with your
`split_heads` and compares, layer by layer.

In [ ]:
def attention(q, k, v):
    """One attention step, softmax(q kᵀ / √d_k) v, for one head or a stack of them."""
    weights = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1]), dim=-1)
    return weights, weights @ v

In [ ]:
# Setup: run this cell, no need to read it.
# It records each encoder layer's input X, and each head's self-attention weights as the model
# computed them, while the model reads the sentence. (This model normalizes after attention,
# so X is exactly what its self-attention receives.)
from torchlingo.models.transformer_simple import capture_self_attention

encoder_layers = transformer.transformer.encoder.layers
layer_inputs = []
hooks = [
    layer.register_forward_pre_hook(
        lambda layer, args, kwargs: layer_inputs.append((args[0] if args else kwargs["src"])[0]),
        with_kwargs=True,
    )
    for layer in encoder_layers
]
try:
    with capture_self_attention(transformer.transformer.encoder, average_heads=False) as captured:
        with torch.no_grad():
            transformer.encode(source)
finally:
    for hook in hooks:
        hook.remove()
model_weights = [w[0] for w in captured]   # per layer: (heads, pieces, pieces)

In [ ]:
layer_weights = []
for number, (layer, X) in enumerate(zip(encoder_layers, layer_inputs), start=1):
    attn = layer.self_attn
    W_Q, W_K, W_V = attn.in_proj_weight.split(attn.embed_dim)   # each (d_model, d_model)
    b_Q, b_K, b_V = attn.in_proj_bias.split(attn.embed_dim)
    with torch.no_grad():
        Q = split_heads(X @ W_Q.T + b_Q, attn.num_heads)   # (heads, pieces, d_k)
        K = split_heads(X @ W_K.T + b_K, attn.num_heads)
        V = split_heads(X @ W_V.T + b_V, attn.num_heads)
        weights, _ = attention(Q, K, V)
    gap = (weights - model_weights[number - 1]).abs().max().item()
    print(f"layer {number}: largest difference from the model's own weights {gap:.1e}")
    assert gap < 1e-4, f"layer {number} differs from the model: recheck split_heads against Step 1"
    layer_weights.append(weights)

print("Your heads are the model's, in every layer.")

## Step 3: predict, then read the first layer head by head

The cell after the prediction prints each of the first layer's four heads, then the four
averaged. Rows and columns are both the sentence's pieces: each row is a piece, and the darker
the cell (`·` `░` `▒` `▓` `█`), the more weight it puts on the piece in that column.

**Write a prediction first**, in `PREDICTION` below, in a sentence or two: will the four heads
look at the same pieces or different ones; will their average be sharper or more spread out than
a single head; and will the deeper layers be sharper or more spread out than the first? The
notebook stops there until you write it. Step 4 measures all three.

In [ ]:
PREDICTION = ""   # <- your prediction, in a sentence or two

assert PREDICTION.strip(), "Write your prediction in PREDICTION, then run this cell again."
print("Prediction recorded.")

In [ ]:
first_layer = layer_weights[0]
for head, weights in enumerate(first_layer, start=1):
    print(f"First layer, head {head}")
    print(format_attention(weights, source_pieces, source_pieces))
    print()
print("First layer, the four heads averaged")
print(format_attention(first_layer.mean(dim=0), source_pieces, source_pieces))

## Step 4: how sharp is each layer?

A way to put one number on what you just looked at: for each row, take its largest weight, and
average those over the rows. 1.0 means every piece puts all its weight on a single piece; with
13 pieces, 1/13 ≈ 0.08 means every piece spreads its weight evenly.

**Yours to write: `sharpness`.** It takes weights shaped (..., rows, columns), where the `...`
may be nothing or a dimension of heads, and returns one number per map. Tensors have `.max` and
`.mean`, each taking a `dim`; check in the PyTorch documentation what `.max(dim=...)` returns,
because it is not just the maxima. Work out the second and third check cases on paper first. The
cell then prints the measurement for every head of every layer, and for each layer's average,
beside your prediction.

In [ ]:
def sharpness(weights):
    """Return each row's largest weight, averaged over the rows, for every map in weights.

    TODO: write it. weights is (..., rows, columns), each row summing to 1; the result has
    shape (...): one number per map.
    """
    raise NotImplementedError("write sharpness, then run this cell again")


# Cases to work on paper first.
assert torch.isclose(sharpness(torch.eye(3)), torch.tensor(1.0)), (
    "every row puts all its weight on one column, so the sharpness is 1.0"
)
assert torch.isclose(sharpness(torch.full((4, 4), 0.25)), torch.tensor(0.25)), (
    "every row spreads evenly over four columns, so the sharpness is 0.25"
)
uneven = torch.tensor([[1.0, 0.0, 0.0], [0.5, 0.5, 0.0]])
assert torch.isclose(sharpness(uneven), torch.tensor(0.75)), (
    f"rows' largest weights are 1.0 and 0.5, so 0.75; got {float(sharpness(uneven)):.2f}. "
    "Take the largest weight in each row, not in each column."
)
two_maps = torch.stack([torch.eye(2), torch.full((2, 2), 0.5)])
assert tuple(sharpness(two_maps).shape) == (2,), "a stack of two maps gives two numbers"
print("sharpness passes its checks.")
print()

print("Your prediction:", PREDICTION.strip())
print()
print("Sharpness: each row's largest weight, averaged over the rows")
for number, weights in enumerate(layer_weights, start=1):
    heads = "  ".join(f"{s:.2f}" for s in sharpness(weights))
    print(f"  layer {number}: heads {heads}   averaged {float(sharpness(weights.mean(dim=0))):.2f}")

**Before you read on**, write one sentence in `MISSED`: which part of your prediction was wrong,
and what in the output showed it. If none was, say which part came closest.

In [ ]:
MISSED = ""   # <- which part of your prediction was wrong, and what in the output showed it

print("What missed:", MISSED.strip() or "(none written)")

## Read it head by head

**Each head is sharp; the heads together are not.** In the first layer nearly every row of every
head is a single `███`: each piece puts almost all its attention on one other piece. Average the
four heads and those peaks fall to `▒▒▒` and `░░░`. The cross-attention map in Lecture 8b's
notebook was an average too, so its spread-out look says as much about averaging as about the
model.

**Sharp is not the same as readable.** A few rows tell a story. Head 2 joins the two pieces of
"report": `▁re` looks at `port`, and `port` looks back at `▁re`. In head 4, `▁This`, `▁is` and
`▁our` each look at themselves. Most rows tell none: in head 1, `port` looks at `<sos>` and
`<sos>` looks at `▁ma`. Nothing trained these heads to do anything a person would name. Each
learned whatever lowered the translation loss, so reading a story into every row is a mistake,
and so is concluding from one tidy row that a head "does" something.

**The two copies of "report" get the same rows.** "report" appears twice, at different
positions, and in every head of the first layer both copies attend in almost exactly the same
way. At this depth the encoder is still treating them as the same word.

**Deeper layers spread out.** The sharpness lines show it: near 1.00 for each head in the first
layer, then well under half in the layers after it, where each piece draws on several others at
once. That is where context gets mixed in, and it is also why the deeper maps are harder to read
than the first.